# Similarity-Based Models Experiment

Implementation and comparison of different approaches to recommend movies based on similarity

Plan:
1. Start with simple models (Genres matching vs. User Ratings matching)
2. Improve Content-Based by using movie titles (TF-IDF)
3. Mix both approaches to see if they help each other
4. Check metrics for top-10, top-20, and top-50 recommendations

In [75]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

# adding src to path
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data import loader, splitter
from src.evaluation import metrics
from src.models.content_based import ContentBasedRecommender
from src.models.collaborative import ItemItemRecommender

## Setup
Using the team's data loader and splitter.
We split data by time (Temporal Split), because in real life we want to predict future interactions based on past ones

In [76]:
movies = loader.load_movies()
ratings = loader.load_ratings()


train_df, val_df, test_df = splitter.split_temporal(
    dataframe=ratings,
    train_ratio=0.8,
    validation_ratio=0.1,
    test_ratio=0.1
)

full_test_df = pd.concat([val_df, test_df])

train_pivot = train_df.pivot(
    index='user_id',
    columns='item_id',
    values='rating'
).fillna(0)


print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Train pivot shape: {train_pivot.shape}")

Loaded 3883 movies from /Users/markmatviyiv/studyspace/Recommender Systems/ucu-recsys/data/ml-1m/movies.dat
Loaded 1000209 ratings from /Users/markmatviyiv/studyspace/Recommender Systems/ucu-recsys/data/ml-1m/ratings.dat
Train shape: (800244, 4)
Validation shape: (99950, 4)
Test shape: (100015, 4)
Train pivot shape: (6040, 3667)


## Helper functions
To compare models fairly, calculating ranking metrics (NDCG, Precision, Recall) for different list sizes (K=10, 20, 50)

In [77]:
all_results = []

def evaluate_model(model, model_name, test_df, train_pivot, n_users=100, k_list=[10, 20, 50]):
    test_users = test_df['user_id'].unique()[:n_users]
    subset_test = test_df[test_df['user_id'].isin(test_users)]
    
    recs_list = []
    max_k = max(k_list)
    
    is_cb = isinstance(model, ContentBasedRecommender)
    
    for user_id in tqdm(test_users, desc=f"Evaluating {model_name}", leave=False):
        if is_cb:
            user_recs = model.recommend(user_id, train_pivot, top_k=max_k)
        else:
            user_recs = model.recommend(user_id, top_k=max_k)
            
        if not user_recs.empty:
            recs_list.append(user_recs)
            
    if not recs_list:
        return
        
    all_recs = pd.concat(recs_list)
    
    for k in k_list:
        metrics_res = metrics.compute_ranking_metrics(subset_test, all_recs, top_k=k)
        metrics_res['Model'] = model_name
        metrics_res['K'] = k
        all_results.append(metrics_res)
        
    return


In [78]:
def display_results(results_list):
    if not results_list: return
    df = pd.DataFrame(results_list)
    
    k_values = sorted(df['K'].unique())
    
    for k in k_values:
        print(f"\n--- Metrics @ K={k} ---")
        subset = df[df['K'] == k].drop(columns=['K']).set_index('Model')
        cols = ['ndcg', 'map', 'precision', 'recall']
        subset = subset[cols]
        display(subset.style.format("{:.4f}"))

## Training and evaluation

Testing 4 variations:
- CB (Jaccard): simplest approach, Movies are similar if they share genres
- CB (TF-IDF): smarter approach, using movie titles + genres to find specific connections (like sequels)
- CF (Cosine): classical collaborative filtering
- CF (Pearson): same as Cosine but tries to account for user rating bias

In [ ]:
# Content-Based (Jaccard)
cb_jaccard = ContentBasedRecommender(similarity_method='jaccard')
cb_jaccard.fit(movies)
evaluate_model(cb_jaccard, "CB (Jaccard)", full_test_df, train_pivot)

# Content-Based (TF-IDF title+genre)
cb_tfidf = ContentBasedRecommender(similarity_method='tfidf')
cb_tfidf.fit(movies)
evaluate_model(cb_tfidf, "CB (TF-IDF)", full_test_df, train_pivot)

# CF (Cosine)
cf_cosine = ItemItemRecommender(method='cosine')
cf_cosine.fit(train_df)
evaluate_model(cf_cosine, "CF (Cosine)", full_test_df, train_pivot)

# CF (Pearson)
cf_pearson = ItemItemRecommender(method='pearson')
cf_pearson.fit(train_df)
evaluate_model(cf_pearson, "CF (Pearson)", full_test_df, train_pivot)

In [80]:
results_df = pd.DataFrame(all_results)
display_results(all_results)


--- Metrics @ K=10 ---


,ndcg,map,precision,recall
Model,,,,
CB (Jaccard),0.0339,0.0139,0.0320,0.0131
CB (TF-IDF),0.0306,0.0116,0.0290,0.0170
CF (Cosine),0.1488,0.0719,0.1380,0.0865
CF (Pearson),0.1291,0.0624,0.1160,0.0609



--- Metrics @ K=20 ---


,ndcg,map,precision,recall
Model,,,,
CB (Jaccard),0.0307,0.0099,0.0250,0.0213
CB (TF-IDF),0.0385,0.0116,0.0330,0.0319
CF (Cosine),0.1543,0.0629,0.1135,0.1334
CF (Pearson),0.1208,0.0457,0.0895,0.0966



--- Metrics @ K=50 ---


,ndcg,map,precision,recall
Model,,,,
CB (Jaccard),0.0371,0.0092,0.0192,0.0516
CB (TF-IDF),0.0567,0.0154,0.0338,0.0777
CF (Cosine),0.1815,0.0624,0.0852,0.2328
CF (Pearson),0.1477,0.0452,0.0744,0.1933


## Hybrid model

Collaborative filtering is usually strong but sparse. Content-Based covers everything but is generic

So here is a try to combine them using a weighted sum: $0.8 \times CF + 0.2 \times CB$

In [81]:
def evaluate_hybrid(cf_model, cb_model, test_df, train_pivot, alpha=0.8, n_users=100, k_list=[10, 20, 50]):
    test_users = test_df['user_id'].unique()[:n_users]
    subset_test = test_df[test_df['user_id'].isin(test_users)]
    recs_list = []
    max_k = max(k_list)
    
    for user_id in tqdm(test_users, desc="Evaluating Hybrid", leave=False):
        if user_id not in train_pivot.index: continue
        user_ratings = train_pivot.loc[user_id]
        rated_items = user_ratings[user_ratings > 0].index
        
        common_items = cf_model.sim_df.index.intersection(cb_model.sim_df.index)
        
        # CF Scores
        valid_rated_cf = user_ratings.index.intersection(cf_model.sim_df.index)
        if len(valid_rated_cf) > 0:
            scores_cf = user_ratings[valid_rated_cf].values.reshape(1,-1).dot(
                cf_model.sim_df.loc[valid_rated_cf, common_items].values).flatten()
        else:
            scores_cf = np.zeros(len(common_items))
            
        # CB Scores
        valid_rated_cb = user_ratings.index.intersection(cb_model.sim_df.index)
        if len(valid_rated_cb) > 0:
            scores_cb = user_ratings[valid_rated_cb].values.reshape(1,-1).dot(
                cb_model.sim_df.loc[valid_rated_cb, common_items].values).flatten()
        else:
            scores_cb = np.zeros(len(common_items))
            
        # Mix
        final_scores = alpha * scores_cf + (1 - alpha) * scores_cb
        
        scores_series = pd.Series(final_scores, index=common_items)
        scores_series = scores_series.drop(index=rated_items, errors='ignore')
        top_items = scores_series.nlargest(max_k)
        
        recs_list.append(pd.DataFrame({'user_id': user_id, 'item_id': top_items.index, 'prediction': top_items.values}))
        
    if not recs_list: return
    all_recs = pd.concat(recs_list)
    
    for k in k_list:
        metrics_res = metrics.compute_ranking_metrics(subset_test, all_recs, top_k=k)
        metrics_res['Model'] = f"Hybrid (alpha={alpha})"
        metrics_res['K'] = k
        all_results.append(metrics_res)


In [82]:
evaluate_hybrid(cf_cosine, cb_tfidf, full_test_df, train_pivot)
display_results(all_results)


--- Metrics @ K=10 ---


,ndcg,map,precision,recall
Model,,,,
CB (Jaccard),0.0339,0.0139,0.0320,0.0131
CB (TF-IDF),0.0306,0.0116,0.0290,0.0170
CF (Cosine),0.1488,0.0719,0.1380,0.0865
CF (Pearson),0.1291,0.0624,0.1160,0.0609
Hybrid (alpha=0.8),0.1493,0.0720,0.1360,0.0861



--- Metrics @ K=20 ---


,ndcg,map,precision,recall
Model,,,,
CB (Jaccard),0.0307,0.0099,0.0250,0.0213
CB (TF-IDF),0.0385,0.0116,0.0330,0.0319
CF (Cosine),0.1543,0.0629,0.1135,0.1334
CF (Pearson),0.1208,0.0457,0.0895,0.0966
Hybrid (alpha=0.8),0.1569,0.0652,0.1165,0.1298



--- Metrics @ K=50 ---


,ndcg,map,precision,recall
Model,,,,
CB (Jaccard),0.0371,0.0092,0.0192,0.0516
CB (TF-IDF),0.0567,0.0154,0.0338,0.0777
CF (Cosine),0.1815,0.0624,0.0852,0.2328
CF (Pearson),0.1477,0.0452,0.0744,0.1933
Hybrid (alpha=0.8),0.1850,0.0636,0.0876,0.2362


# Final conclusions

Some thoughts after running these experiments:

1. Collaborative Filtering is the winner:\
   The simple Item-Item Cosine model gave the best results (NDCG ~0.15). This makes sense for MovieLens because the dataset is dense enough. User behavior patterns are just much stronger signals than simple genre tags

2. Content-Based is weak, but titles help:\
   Using just genres (Jaccard) is too broad. Adding TF-IDF on titles didn't improve the top ranking - NDCG dropped slightly, but it did improve finding relevant items deeper in the list (Recall)\
   This means it helps find specific things like sequels, even if it's not perfect at ranking them at the very top.

3. Hybrid didn't help:\
   Mixing them ($0.8 \cdot CF + 0.2 \cdot CB$) actually hurt the performance slightly compared to pure CF. This suggests that for users who already have history, the Content-Based signal is just adding noise\
   It would probably be better to use Content-Based only for new users aka Cold Start, rather than mixing it for everyone